# Dijkstra and A*: worked lecture notebook

All outputs are computed from the accompanying code. Compare costs rather than one path when ties exist.

In [1]:
# Works when launched from the repository root or notebooks/.
import os, sys
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "algorithms").exists():
    ROOT = ROOT.parent
assert (ROOT / "algorithms").exists(), "Open this notebook inside the repository"
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
from heapq import heappop, heappush
from itertools import count
from math import inf
from algorithms.dijkstra import IndexedMinPQ
from algorithms.common import recover
from graphs.main_graph import GRAPH, HEURISTIC



## 09-16 min — Bruteforce

Enumerate simple routes. Why must the visited set be path-local?

In [2]:
def search(graph, start, target):
    best_cost, best_path = inf, []
    def visit(u, cost, path, on_path):
        nonlocal best_cost, best_path
        if u == target:
            if cost < best_cost:
                best_cost, best_path = cost, path.copy()
            return
        for v, weight in graph[u]:
            if v not in on_path:
                visit(v, cost + weight, path + [v], on_path | {v})
    visit(start, 0, [start], {start})
    return best_cost, best_path

print(search(GRAPH, 'S', 'T'))


(8, ['S', 'A', 'B', 'C', 'T'])


## 28-34 min — Lazy Relaxation

Fill the accumulated cost and improvement condition.

In [3]:
def search(graph, start, target):
    dist, parent = {start: 0}, {}
    ticket = count()
    queue = [(0, next(ticket), start)]
    while queue:
        g, _, u = heappop(queue)
        if g != dist[u]:
            continue
        if u == target:
            return g, recover(parent, start, target)
        for v, weight in graph[u]:
            candidate = g + weight
            if candidate < dist.get(v, inf):
                dist[v] = candidate
                parent[v] = u
                heappush(queue, (candidate, next(ticket), v))
    return inf, []

print(search(GRAPH, 'S', 'T'))


(8, ['S', 'A', 'B', 'C', 'T'])


## 34-38 min — Stale And Parent

Skip obsolete records; replace the predecessor on every improvement.

In [4]:
def search(graph, start, target):
    dist, parent = {start: 0}, {}
    ticket = count()
    queue = [(0, next(ticket), start)]
    while queue:
        g, _, u = heappop(queue)
        if g != dist[u]:
            continue
        if u == target:
            return g, recover(parent, start, target)
        for v, weight in graph[u]:
            candidate = g + weight
            if candidate < dist.get(v, inf):
                dist[v] = candidate
                parent[v] = u
                heappush(queue, (candidate, next(ticket), v))
    return inf, []

print(search(GRAPH, 'S', 'T'))


(8, ['S', 'A', 'B', 'C', 'T'])


## 38-41 min — Goal Check

The solution below checks extraction. Explain why generation is too early.

In [5]:
def search(graph, start, target):
    dist, parent = {start: 0}, {}
    ticket = count()
    queue = [(0, next(ticket), start)]
    while queue:
        g, _, u = heappop(queue)
        if g != dist[u]:
            continue
        if u == target:
            return g, recover(parent, start, target)
        for v, weight in graph[u]:
            candidate = g + weight
            if candidate < dist.get(v, inf):
                dist[v] = candidate
                parent[v] = u
                heappush(queue, (candidate, next(ticket), v))
    return inf, []

print(search(GRAPH, 'S', 'T'))


(8, ['S', 'A', 'B', 'C', 'T'])


## 41-48 min — Eager Queue

Keep one queue entry per frontier node.

In [6]:
def search(graph, start, target):
    dist, parent = {start: 0}, {}
    queue = IndexedMinPQ()
    queue.insert(start, 0)
    while queue:
        g, u = queue.pop_min()
        if u == target:
            return g, recover(parent, start, target)
        for v, weight in graph[u]:
            candidate = g + weight
            if candidate < dist.get(v, inf):
                dist[v] = candidate
                parent[v] = u
                if v in queue:
                    queue.decrease_key(v, candidate)
                else:
                    queue.insert(v, candidate)
    return inf, []

print(search(GRAPH, 'S', 'T'))


(8, ['S', 'A', 'B', 'C', 'T'])


## 68-77 min — Astar

Change priority, not the meaning of the stored route cost.

In [7]:
def search(graph, start, target, h):
    dist, parent = {start: 0}, {}
    ticket = count()
    queue = [(h[start], next(ticket), 0, start)]
    while queue:
        f, _, g, u = heappop(queue)
        if g != dist[u]:
            continue
        if u == target:
            return g, recover(parent, start, target)
        for v, weight in graph[u]:
            candidate = g + weight
            if candidate < dist.get(v, inf):
                dist[v] = candidate
                parent[v] = u
                priority = candidate + h[v]
                heappush(queue, (priority, next(ticket), candidate, v))
    return inf, []

print(search(GRAPH, 'S', 'T', HEURISTIC))


(8, ['S', 'A', 'B', 'C', 'T'])


## 77-86 min — comparative experiment

Before running: predict which claims are guarantees and which are empirical. This notebook provides a ready-to-use substitute while the separate closing-demo application is deferred.

In [8]:
from algorithms import astar, dijkstra_lazy
from algorithms.experiments import astar_closed_bug, stop_on_generation_bug
from graphs.fixtures import reopening_case, grid_case
case = reopening_case()
for fn in (astar_closed_bug, astar):
    r = fn(case['graph'], 'S', 'T', case['h'])
    print(fn.__name__, 'cost=', r.distance, 're-expansions=', r.stats.reexpansions)
print('Stop-on-generation bug:', stop_on_generation_bug(GRAPH, 'S', 'T'))
for kind in ('zero', 'euclidean', 'manhattan'):
    case = grid_case(24, 16, True, kind)
    r = astar(case['graph'], case['start'], case['target'], case['h'])
    print(kind, 'cost=', r.distance, 'expansions=', r.stats.expansions)


astar_closed_bug cost= 6 re-expansions= 0
astar cost= 5 re-expansions= 1
Stop-on-generation bug: 15
zero cost= 33 expansions= 315
euclidean cost= 33 expansions= 269
manhattan cost= 33 expansions= 224
